In [ ]:
# Following example from https://pub.towardsai.net/efficient-camera-stream-with-python-b6adf93fab32

import platform
import subprocess
import time
import cv2
import numpy as np

# Video Stream Class
class VideoStreamCV:
    def __init__(self, src: int, fps: int, resolution: tuple[int, int]):
        self.src = src
        self.fps = fps
        self.resolution = resolution
        self.cap = self._open_camera()
        self.wait_for_cam()
        
    def _open_camera(self):
        cap = cv2.VideoCapture(self.src)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.resolution[0])
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.resolution[1])
        fourcc = cv2.VideoWriter_fourcc(*"MJPG")
        cap.set(cv2.CAP_PROP_FOURCC, fourcc)
        cap.set(cv2.CAP_PROP_FPS, self.fps)
        return cap
    
    def read(self):
        ret, frame = self.cap.read()
        if not ret:
            return None
        return frame
    
    def release(self):
        self.cap.release()
        
    def wait_for_cam(self):
        for _ in range(30):
            frame = self.read()
        if frame is not None:
            return True
        return False

In [13]:
# Following example for testing read time from https://pub.towardsai.net/efficient-camera-stream-with-python-b6adf93fab32

def timeit(func):
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        print(f"Main function time: {round(t1-t0, 4)}s")
        return result
    
    return wrapper

def computation_task():
    for _ in range(5000000):
        9999 * 9999

@timeit
def run(cam: VideoStreamCV, run_task: bool):
    timer = []
    for _ in range(100):
        t0 = time.perf_counter()
        cam.read()
        timer.append(time.perf_counter() - t0)

        if run_task:
            computation_task()

    cam.release()
    return round(np.mean(timer), 4)


fps = 30
resolution = (1920, 1080)

for run_task in [False, True]:
    cam = VideoStreamCV(src=0, fps=fps, resolution=resolution)
    print(f"CV2, task {run_task}:")
    print(f"Mean frame read time: {run(cam=cam, run_task=run_task)}s\n")

CV2, task False:
Main function time: 3.5749s
Mean frame read time: 0.0344s

CV2, task True:
Main function time: 8.435s
Mean frame read time: 0.0126s



In [ ]:
# Displaying video capture for testing
# First, lets see how stack does it:

# Source - https://stackoverflow.com/a/11449901
# Posted by sastanin, modified by community. See post 'Timeline' for change history
# Retrieved 2026-11-08, License - CC BY-SA 3.0

# cv2.namedWindow("preview")
# vc = cv2.VideoCapture(0)

# if vc.isOpened(): # try to get the first frame
#     rval, frame = vc.read()
# else:
#     rval = False

# while rval:
#     cv2.imshow("preview", frame)
#     rval, frame = vc.read()
#     key = cv2.waitKey(20)
#     if key == 27: # exit on ESC
#         break

# cv2.destroyWindow("preview")
# vc.release()


# Now, lets do it using our architecture:

cam = VideoStreamCV(src=0, fps=fps, resolution=resolution)
if cam.cap.isOpened():
    rval, frame = cam.cap.read()
else:
    rval = False
    
while rval:
    cv2.imshow("preview", frame)
    rval, frame = cam.cap.read()
    key = cv2.waitKey(20)
    if key == 27: #exit on ESC
        break
    
cv2.destroyWindow("preview")
cam.cap.release()
